In [2]:
# bert_layer_probe_and_train.py
#
# WHAT THIS DOES
# ══════════════
# Phase 1 — Layer Probing
#   Attach a tiny linear classifier (probe) to the mean-pooled output of
#   every BERT layer (0 through N-1 plus the embedding layer).
#   Train each probe independently for a few epochs with BERT fully frozen.
#   Measure rare-class macro-F1 per layer on the dev set.
#   → Tells you exactly which layer encodes the most discriminative
#     information for rare rhetorical roles.
#
# Phase 2 — Layer-Selective Main Training
#   Use the probing results to build a custom freeze/LR schedule:
#     • The single best layer gets the highest LR (BEST_LAYER_LR_MULTIPLIER × BERT_LR)
#     • The two layers adjacent to it get a moderate boost
#     • All other layers below the global freeze threshold stay frozen
#     • Layers above the freeze threshold train at their normal decayed LR
#   This lets the model concentrate learning where representations are already
#   most useful for rare classes, rather than fine-tuning uniformly.
#
# USAGE
# ═════
#   python bert_layer_probe_and_train.py --phase probe   # runs Phase 1 only
#   python bert_layer_probe_and_train.py --phase train   # runs Phase 2 only
#   python bert_layer_probe_and_train.py                 # runs both end-to-end
#
#   After Phase 1, results are written to:
#     probe_results/layer_probe_results.json
#     probe_results/layer_rare_f1_chart.png
#
#   Phase 2 reads probe_results/layer_probe_results.json automatically.
#   If the file does not exist, Phase 2 falls back to a sensible default.

import os, json, random, argparse, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG — shared
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH   = "dataset/build_train.jsonl"
DEV_PATH     = "dataset/build_dev.jsonl"
TEST_PATH    = "dataset/build_test.jsonl"
PROBE_DIR    = "probe_results"
MAIN_OUT_DIR = "rrc_layer_selective_logs"
BEST_MODEL_DIR = os.path.join(MAIN_OUT_DIR, "best_model")

for d in [PROBE_DIR, MAIN_OUT_DIR, BEST_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

SEED           = 42
MAX_SEQ_LENGTH = 32
RARE_THRESHOLD = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# ═══════════════════════════════════════════════════════════
# CONFIG — Phase 1 (probing)
# ═══════════════════════════════════════════════════════════
PROBE_EPOCHS     = 8       # small; BERT is frozen so probes train fast
PROBE_LR         = 1e-3
PROBE_BATCH      = 32      # can be large because BERT is frozen
PROBE_WEIGHT_DECAY = 1e-4

# ═══════════════════════════════════════════════════════════
# CONFIG — Phase 2 (main training)
# ═══════════════════════════════════════════════════════════
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 2e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.3
BERT_LR_DECAY   = 0.9

# Layer-selective LR multipliers (applied on top of BERT_LR)
BEST_LAYER_LR_MULTIPLIER     = 5.0   # best probe layer gets 5× BERT_LR
ADJACENT_LAYER_LR_MULTIPLIER = 2.5   # ±1 neighbour layers get 2.5×

# How many top layers are always unfrozen regardless of probe result
ALWAYS_UNFREEZE_TOP_N = 2   # layers (N-2) and (N-1) always train

# Global freeze floor — layers below this index stay frozen no matter what
# (override by probe: if best layer is below this, it still gets unfrozen)
BASE_FREEZE_BELOW = 6

# Architecture
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1

AUX_CE_WEIGHT    = 0.5
FOCAL_GAMMA      = 2.0
LABEL_SMOOTHING  = 0.05
RARE_OVERSAMPLE  = 3.0

SWA_START_FRAC   = 0.85
SWA_LR           = 5e-5
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_RATIO     = 0.05


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# DATA UTILITIES
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                out.append(json.loads(s))
    return out


def extract_docs(raw, max_sents=256):
    all_docs = []
    for doc in raw:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    v = item.get("value", {})
                    t = v.get("text", "").strip()
                    l = v.get("labels", ["NONE"])[0]
                    if t:
                        sents.append(t)
                        labs.append(label2id.get(l, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    freqs   = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\nRare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, freqs


def compute_class_weights(docs, device=DEVICE):
    """Inverse-frequency weights for CE loss — strongly upweights rare classes."""
    all_ids = [lid for _, labs in docs for lid in labs]
    counts  = Counter(all_ids)
    total   = len(all_ids)
    weights = []
    for i in range(NUM_LABELS):
        freq = counts.get(i, 0) / total
        weights.append(1.0 / (freq + 1e-6))
    w = torch.tensor(weights, dtype=torch.float32, device=device)
    w = w / w.mean()   # normalise so mean weight = 1
    return w


# ═══════════════════════════════════════════════════════════
# FLAT SENTENCE DATASET  (for probing — one row per sentence)
# ═══════════════════════════════════════════════════════════
class FlatSentenceDataset(Dataset):
    """Each item is a single sentence + its label.  Used for probing."""
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.items     = [(s, l) for sents, labs in docs
                          for s, l in zip(sents, labs)]
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        sent, label = self.items[idx]
        enc = self.tokenizer(
            sent,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])).squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }


# ═══════════════════════════════════════════════════════════
# DOCUMENT DATASET  (for main training)
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs      = docs
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L, B  = batch[0]["input_ids"].shape[1], len(batch)
    ids   = torch.zeros(B, T_max, L, dtype=torch.long)
    mask  = torch.zeros(B, T_max, L, dtype=torch.long)
    types = torch.zeros(B, T_max, L, dtype=torch.long)
    labs  = torch.full((B, T_max), -100, dtype=torch.long)
    lens  = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        ids[i, :t]   = b["input_ids"]
        mask[i, :t]  = b["attention_mask"]
        types[i, :t] = b["token_type_ids"]
        labs[i, :t]  = b["labels"]
        lens[i]      = t
    return ids, mask, types, labs, lens


def make_oversampled_loader(dataset, docs, rare_ids, batch_size):
    """WeightedRandomSampler that oversamples documents containing rare sentences."""
    doc_weights = []
    for sents, labs in docs:
        has_rare = any(l in rare_ids for l in labs)
        doc_weights.append(RARE_OVERSAMPLE if has_rare else 1.0)
    sampler = WeightedRandomSampler(
        weights     = doc_weights,
        num_samples = len(doc_weights),
        replacement = True,
    )
    return DataLoader(dataset, batch_size=batch_size,
                      sampler=sampler, collate_fn=collate_rrc)


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0
        logits_v, targets_v = logits[valid], targets[valid]
        log_p  = F.log_softmax(logits_v, dim=-1)
        log_pt = log_p.gather(1, targets_v.unsqueeze(1)).squeeze(1)
        pt     = log_pt.exp()
        focal  = -((1 - pt) ** self.gamma) * log_pt
        if self.weight is not None:
            w = self.weight[targets_v]
            focal = focal * w
        return focal.mean()


# ═══════════════════════════════════════════════════════════
# PHASE 1 — LAYER PROBE CLASSIFIER
# ═══════════════════════════════════════════════════════════
class LayerProbe(nn.Module):
    """
    A tiny linear probe attached to one BERT layer's mean-pooled output.
    BERT stays completely frozen — only the probe weights update.
    This isolates what each layer already knows about rhetorical roles.
    """
    def __init__(self, hidden_size: int, num_labels: int):
        super().__init__()
        self.fc = nn.Linear(hidden_size, num_labels)

    def forward(self, x):
        return self.fc(x)   # (N, num_labels)


def extract_layer_representations(
    bert_model, tokenizer, docs, layer_idx,
    batch_size=PROBE_BATCH, device=DEVICE
):
    """
    Extract mean-pooled sentence embeddings from a specific BERT layer.
    Returns (N_sentences, hidden_size) tensor and corresponding label tensor.
    """
    bert_model.eval()
    dataset = FlatSentenceDataset(docs, tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_embs, all_labs = [], []

    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            types = batch["token_type_ids"].to(device)
            labs  = batch["label"]

            out = bert_model(
                input_ids      = ids,
                attention_mask = mask,
                token_type_ids = types,
                output_hidden_states = True,
            )
            # hidden_states: tuple of (num_layers+1) tensors, each (B, L, H)
            # index 0 = embedding layer, 1 = after layer 0, etc.
            hidden = out.hidden_states[layer_idx]   # (B, L, H)
            mask_f = mask.unsqueeze(-1).float()
            emb    = (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)
            all_embs.append(emb.cpu())
            all_labs.append(labs)

    return torch.cat(all_embs, dim=0), torch.cat(all_labs, dim=0)


def probe_one_layer(
    bert_model, tokenizer,
    train_docs, dev_docs,
    layer_idx, rare_ids, class_weights,
    device=DEVICE,
):
    """Train a linear probe on layer_idx and return dev rare-class macro-F1."""
    print(f"  Probing layer {layer_idx:2d} ...")

    # Pre-extract embeddings once (no BERT gradient needed)
    train_embs, train_labs = extract_layer_representations(
        bert_model, tokenizer, train_docs, layer_idx, device=device
    )
    dev_embs, dev_labs = extract_layer_representations(
        bert_model, tokenizer, dev_docs, layer_idx, device=device
    )

    hidden_size = train_embs.shape[-1]
    probe = LayerProbe(hidden_size, NUM_LABELS).to(device)
    opt   = torch.optim.Adam(probe.parameters(), lr=PROBE_LR,
                             weight_decay=PROBE_WEIGHT_DECAY)
    ce    = nn.CrossEntropyLoss(weight=class_weights.to(device))

    train_ds = torch.utils.data.TensorDataset(train_embs, train_labs)
    train_ld = DataLoader(train_ds, batch_size=256, shuffle=True)

    best_rare_f1 = 0.0

    for epoch in range(PROBE_EPOCHS):
        probe.train()
        for emb_batch, lab_batch in train_ld:
            emb_batch = emb_batch.to(device)
            lab_batch = lab_batch.to(device)
            logits = probe(emb_batch)
            loss   = ce(logits, lab_batch)
            opt.zero_grad()
            loss.backward()
            opt.step()

        # Dev evaluation
        probe.eval()
        with torch.no_grad():
            dev_logits = probe(dev_embs.to(device))
            dev_preds  = dev_logits.argmax(-1).cpu().tolist()
            dev_true   = dev_labs.tolist()

        present_rare = [r for r in rare_ids if r in dev_true]
        if present_rare:
            rare_f1 = f1_score(dev_true, dev_preds, labels=present_rare,
                               average="macro", zero_division=0)
        else:
            rare_f1 = 0.0

        if rare_f1 > best_rare_f1:
            best_rare_f1 = rare_f1

    macro_f1 = f1_score(dev_true, dev_preds, average="macro", zero_division=0)
    print(f"           layer {layer_idx:2d} | rare_f1={best_rare_f1:.4f} | macro_f1={macro_f1:.4f}")

    return {
        "layer_idx":    layer_idx,
        "rare_f1":      best_rare_f1,
        "macro_f1":     macro_f1,
    }


def run_probing_phase(train_docs, dev_docs, rare_ids, rare_labels):
    """
    Phase 1: probe every BERT layer and save results.
    Returns the index (1-based hidden_states index) of the best layer.
    """
    print("\n" + "="*60)
    print("PHASE 1 — BERT LAYER PROBING")
    print("="*60)
    print(f"Probing {PROBE_EPOCHS} epochs per layer, metric = rare-class macro-F1\n")

    tokenizer  = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    bert_model = AutoModel.from_pretrained(
        INLEGALBERT_MODEL_NAME,
        output_hidden_states=True,
    ).to(DEVICE)

    # Freeze all BERT parameters — probe only
    for p in bert_model.parameters():
        p.requires_grad = False

    class_weights = compute_class_weights(train_docs)
    n_layers      = bert_model.config.num_hidden_layers  # 12 for BERT-base

    # hidden_states indices: 0=embedding, 1=layer0, ..., n_layers=layer(n_layers-1)
    all_results = []
    for idx in range(0, n_layers + 1):   # 0 through 12
        result = probe_one_layer(
            bert_model, tokenizer,
            train_docs, dev_docs,
            layer_idx = idx,
            rare_ids  = rare_ids,
            class_weights = class_weights,
        )
        result["bert_layer_name"] = f"embedding" if idx == 0 else f"layer_{idx-1}"
        all_results.append(result)

    # Sort by rare_f1
    sorted_results = sorted(all_results, key=lambda x: x["rare_f1"], reverse=True)
    best           = sorted_results[0]

    print(f"\n{'='*60}")
    print(f"PROBING RESULTS — best layer for rare classes:")
    print(f"  hidden_states index : {best['layer_idx']}")
    print(f"  BERT layer name     : {best['bert_layer_name']}")
    print(f"  Rare-class macro-F1 : {best['rare_f1']:.4f}")
    print(f"  Macro-F1 (all)      : {best['macro_f1']:.4f}")
    print(f"{'='*60}\n")

    # Save results
    out = {
        "best_layer_hidden_states_idx": best["layer_idx"],
        "best_bert_layer_name":         best["bert_layer_name"],
        "best_rare_f1":                 best["rare_f1"],
        "all_results":                  all_results,
        "rare_labels":                  rare_labels,
        "timestamp":                    datetime.utcnow().isoformat(),
    }
    path = os.path.join(PROBE_DIR, "layer_probe_results.json")
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved probe results to: {path}")

    # Plot
    _plot_probe_results(all_results, rare_labels)

    return best["layer_idx"]


def _plot_probe_results(results, rare_labels):
    names    = [r["bert_layer_name"] for r in results]
    rare_f1s = [r["rare_f1"]  for r in results]
    mac_f1s  = [r["macro_f1"] for r in results]
    best_idx = int(np.argmax(rare_f1s))

    x = np.arange(len(names))
    colors = ["#EF9F27" if i == best_idx else "#85B7EB" for i in range(len(names))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(x, rare_f1s, color=colors, edgecolor="white")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    axes[0].set_title("Rare-class macro-F1 per BERT layer\n(orange = best layer)")
    axes[0].set_ylabel("Rare-class macro-F1")
    axes[0].axhline(max(rare_f1s), color="darkorange", linestyle="--", alpha=0.5)
    axes[0].grid(True, alpha=0.3, axis="y")

    axes[1].bar(x, mac_f1s, color="#B5D4F4", edgecolor="white")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    axes[1].set_title("Overall macro-F1 per BERT layer")
    axes[1].set_ylabel("Macro-F1")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.suptitle(
        f"Layer Probing — InLegalBERT | Rare classes: {rare_labels}",
        fontsize=10,
    )
    plt.tight_layout()
    p = os.path.join(PROBE_DIR, "layer_rare_f1_chart.png")
    plt.savefig(p, dpi=150)
    plt.close()
    print(f"Saved probe chart to: {p}")


# ═══════════════════════════════════════════════════════════
# PHASE 2 — ARCHITECTURE COMPONENTS
# ═══════════════════════════════════════════════════════════

class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query     = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        w = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            w = w.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        w       = self.attn_drop(F.softmax(w, dim=-1))
        context = torch.matmul(w, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


class InLegalBERT_LayerSelective(nn.Module):
    """
    Same BiLSTM → MHA-pool → BiLSTM → CRF architecture as v3,
    but the BERT freeze/LR schedule is driven by probing results.

    Key change: _apply_layer_selective_freeze() reads the best_layer_idx
    (hidden_states index) from probing and ensures that layer stays
    trainable, even if it falls below BASE_FREEZE_BELOW.
    """

    def __init__(
        self,
        best_layer_hs_idx: int,   # hidden_states index of best probe layer
        bert_model_name   = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden  = SENT_LSTM_HIDDEN,
        sent_lstm_layers  = SENT_LSTM_LAYERS,
        ctx_lstm_hidden   = CTX_LSTM_HIDDEN,
        ctx_lstm_layers   = CTX_LSTM_LAYERS,
        mha_heads         = MHA_HEADS,
        mha_dropout       = MHA_DROPOUT,
        num_labels        = NUM_LABELS,
        dropout           = DROPOUT,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)

        # best_layer_hs_idx is a hidden_states index (0=emb, 1=L0, ..., 13=L11)
        # Convert to BERT encoder layer index (0-based): hs_idx - 1
        self.best_hs_idx    = best_layer_hs_idx
        self.best_bert_lidx = max(0, best_layer_hs_idx - 1)  # encoder layer index

        self._apply_layer_selective_freeze()

        self.sent_bilstm = nn.LSTM(
            self.bert_dim, sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out = sent_lstm_hidden * 2

        self.mha_pooling     = MultiHeadAttentionPooling(sent_out, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(sent_out)

        self.ctx_bilstm = nn.LSTM(
            sent_out, ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out = ctx_lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out, ctx_out // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def _apply_layer_selective_freeze(self):
        """
        Freeze strategy driven by probing result:

          1. Always freeze embedding layer.
          2. Freeze encoder layers 0 .. (BASE_FREEZE_BELOW - 1).
          3. Always unfreeze the top ALWAYS_UNFREEZE_TOP_N layers.
          4. Crucially: also unfreeze best_bert_lidx and its immediate
             neighbours (±1), even if they sit below BASE_FREEZE_BELOW.
             This is the direct translation of "make the best layer work harder".
        """
        n_layers     = len(self.bert.encoder.layer)
        best         = self.best_bert_lidx
        always_train = set(range(n_layers - ALWAYS_UNFREEZE_TOP_N, n_layers))
        probe_train  = {max(0, best - 1), best, min(n_layers - 1, best + 1)}
        trainable_set = always_train | probe_train
        # Also unfreeze all layers at or above BASE_FREEZE_BELOW
        for i in range(BASE_FREEZE_BELOW, n_layers):
            trainable_set.add(i)

        # Freeze embeddings
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False

        trained, frozen = [], []
        for i, layer in enumerate(self.bert.encoder.layer):
            if i in trainable_set:
                for p in layer.parameters():
                    p.requires_grad = True
                trained.append(i)
            else:
                for p in layer.parameters():
                    p.requires_grad = False
                frozen.append(i)

        print(f"\nLayer-selective freeze (probe best = layer {best}):")
        print(f"  Frozen  BERT layers : {frozen}")
        print(f"  Trainable BERT layers : {trained}")
        print(f"  (probe-boosted layers : {sorted(probe_train)})\n")

        self.trainable_layer_ids = trained
        self.probe_boosted       = sorted(probe_train)

    def build_optimizer(self):
        """
        Per-layer LR schedule:
          - probe-best layer     : BERT_LR × BEST_LAYER_LR_MULTIPLIER
          - adjacent ±1 layers   : BERT_LR × ADJACENT_LAYER_LR_MULTIPLIER
          - other trainable BERT : BERT_LR × decay^depth
          - head (BiLSTM etc.)   : HEAD_LR
        """
        n_layers     = len(self.bert.encoder.layer)
        best         = self.best_bert_lidx
        adj          = {max(0, best - 1), min(n_layers - 1, best + 1)} - {best}
        param_groups = []

        # Pooler
        param_groups.append({
            "params": list(self.bert.pooler.parameters()),
            "lr":     BERT_LR,
            "weight_decay": WEIGHT_DECAY,
            "name": "bert_pooler",
        })

        for i in self.trainable_layer_ids:
            params = [p for p in self.bert.encoder.layer[i].parameters()
                      if p.requires_grad]
            if not params:
                continue

            if i == best:
                lr = BERT_LR * BEST_LAYER_LR_MULTIPLIER
                tag = f"bert_layer_{i}_PROBE_BEST"
            elif i in adj:
                lr = BERT_LR * ADJACENT_LAYER_LR_MULTIPLIER
                tag = f"bert_layer_{i}_PROBE_ADJ"
            else:
                depth = (n_layers - 1) - i
                lr    = BERT_LR * (BERT_LR_DECAY ** depth)
                tag   = f"bert_layer_{i}"

            param_groups.append({
                "params": params,
                "lr":     lr,
                "weight_decay": WEIGHT_DECAY,
                "name": tag,
            })

        # Head
        head_params = (
            list(self.sent_bilstm.parameters())
            + list(self.mha_pooling.parameters())
            + list(self.sent_layer_norm.parameters())
            + list(self.ctx_bilstm.parameters())
            + list(self.classifier.parameters())
            + list(self.crf.parameters())
        )
        param_groups.append({
            "params": head_params,
            "lr":     HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
            "name": "head",
        })

        print("Optimizer LR schedule:")
        for g in param_groups:
            print(f"  {g['name']:<35} lr={g['lr']:.2e}")

        return torch.optim.AdamW(param_groups)

    # ── encode sentences ─────────────────────────────────
    def encode_sentences(self, input_ids, attention_mask, token_type_ids):
        B, T, L = input_ids.shape
        N       = B * T
        fids    = input_ids.view(N, L)
        fmask   = attention_mask.view(N, L)
        ftypes  = token_type_ids.view(N, L)
        valid   = fmask.sum(-1) > 0

        embs_all = fids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = fids[valid],
                attention_mask = fmask[valid],
                token_type_ids = ftypes[valid],
            )
            embs_all[valid] = out.last_hidden_state.to(embs_all.dtype)

        embs_all    = self.dropout(embs_all)
        lstm_out, _ = self.sent_bilstm(embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (fmask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None,
                class_weights=None, focal_loss_fn=None):
        sv   = self.dropout(
            self.encode_sentences(input_ids, attention_mask, token_type_ids)
        )
        if lengths is not None:
            packed    = nn.utils.rnn.pack_padded_sequence(
                sv, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_o, _ = self.ctx_bilstm(packed)
            ctx, _    = nn.utils.rnn.pad_packed_sequence(packed_o, batch_first=True)
        else:
            ctx, _ = self.ctx_bilstm(sv)

        ctx       = self.dropout(ctx)
        emissions = self.classifier(ctx)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        B, T, _ = emissions.shape
        if lengths is not None:
            crf_mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                crf_mask[i, :l] = True
        elif labels is not None:
            crf_mask = (labels != -100)
        else:
            crf_mask = torch.ones(B, T, dtype=torch.bool, device=emissions.device)

        if labels is not None:
            safe = labels.clone()
            safe[safe == -100] = 0

            crf_loss = -self.crf(emissions, safe, mask=crf_mask, reduction="mean")

            flat_em = emissions.reshape(B * T, -1)
            flat_lb = labels.reshape(B * T)

            ce_fn = nn.CrossEntropyLoss(
                weight          = class_weights,
                label_smoothing = LABEL_SMOOTHING,
                ignore_index    = -100,
            )
            ce_loss = ce_fn(flat_em, flat_lb)

            focal_loss = focal_loss_fn(flat_em, flat_lb) if focal_loss_fn else 0.0

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss + 0.3 * focal_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=crf_mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    wp          = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec   = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec   = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    wr          = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc         = accuracy_score(all_trues, all_preds)

    pc_f1  = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    pc_pr  = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    pc_re  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)

    per_class = {
        id2label[i]: {"f1": float(pc_f1[i]), "precision": float(pc_pr[i]), "recall": float(pc_re[i])}
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    rare_f1  = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0) if present_rare else 0.0
    rare_pr  = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0) if present_rare else 0.0
    rare_rec = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0) if present_rare else 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_rep   = classification_report(str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)
    cm        = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec, "weighted_precision": wp,
        "macro_recall": macro_rec, "micro_recall": micro_rec, "weighted_recall": wr,
        "rare_f1": rare_f1, "rare_precision": rare_pr, "rare_recall": rare_rec,
        "per_class_metrics": per_class,
        "accuracy": acc, "cls_report": cls_rep, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = -1.0
        self.counter   = 0
        self.stop      = False

    def step(self, score):
        if score > self.best + self.min_delta:
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PHASE 2 — TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, class_weights, rare_ids, device=DEVICE):
        self.model         = model.to(device)
        self.device        = device
        self.class_weights = class_weights.to(device)
        self.focal_fn      = FocalLoss(gamma=FOCAL_GAMMA,
                                       weight=class_weights.to(device),
                                       ignore_index=-100)
        self.rare_ids      = rare_ids

    def _val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, mask, types, labs, lens in loader:
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                labs, lens       = labs.to(self.device), lens.to(self.device)
                loss, _ = self.model(ids, mask, types, labels=labs, lengths=lens,
                                     class_weights=self.class_weights,
                                     focal_loss_fn=self.focal_fn)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, split_name="dev",
                 measure_time=False, model_override=None):
        m = model_override or self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_time else None

        with torch.no_grad():
            for ids, mask, types, labs, lens in loader:
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                lens = lens.to(self.device)
                decoded, _ = m(ids, mask, types, labels=None, lengths=lens)
                for i, seq in enumerate(decoded):
                    tl = int(lens[i].item())
                    all_preds.extend(seq)
                    all_trues.extend(labs[i, :tl].tolist())

        if measure_time:
            elapsed = time.time() - t0
            print(f"  Inference ({split_name}): {elapsed:.2f}s | "
                  f"{len(all_trues)/elapsed:.0f} sent/s")

        return compute_all_metrics(all_trues, all_preds, self.rare_ids)

    def train(self, train_dataset, dev_dataset, train_docs, tokenizer):
        loader    = make_oversampled_loader(train_dataset, train_docs,
                                            self.rare_ids, BATCH_DOCS)
        optimizer = self.model.build_optimizer()
        total_steps  = len(loader) * NUM_EPOCHS // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        swa_model   = AveragedModel(self.model)
        swa_start   = max(1, int(NUM_EPOCHS * SWA_START_FRAC))
        swa_sched   = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
        swa_active  = False

        es          = EarlyStopping()
        best_f1     = -1.0
        best_state  = None
        history     = []
        t_start     = time.time()
        actual_ep   = 0

        print(f"\nSWA starts at epoch {swa_start}/{NUM_EPOCHS}")

        for epoch in range(1, NUM_EPOCHS + 1):
            actual_ep = epoch
            self.model.train()
            run_loss, n_steps = 0.0, 0
            ep_t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, mask, types, labs, lens) in enumerate(loader):
                ids, mask, types = ids.to(self.device), mask.to(self.device), types.to(self.device)
                labs, lens       = labs.to(self.device), lens.to(self.device)

                loss, _ = self.model(ids, mask, types, labels=labs, lengths=lens,
                                     class_weights=self.class_weights,
                                     focal_loss_fn=self.focal_fn)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()

                run_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active: scheduler.step()
                optimizer.zero_grad()

            if epoch >= swa_start:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_sched.step()

            ep_time     = time.time() - ep_t0
            avg_loss    = run_loss / max(1, n_steps)
            val_loss    = self._val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset)

            swa_tag = " [SWA]" if swa_active else ""
            print(
                f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
                f"train={avg_loss:.4f} | val={val_loss:.4f} | "
                f"macro_f1={val_metrics['macro_f1']:.4f} | "
                f"rare_f1={val_metrics['rare_f1']:.4f} | "
                f"acc={val_metrics['accuracy']:.4f} | "
                f"ES={es.counter}/{es.patience}{swa_tag}"
            )

            history.append({
                "epoch": epoch, "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "swa_active":   swa_active, "epoch_time": ep_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if es.step(val_metrics["macro_f1"]):
                print(f"\nEarly stopping at epoch {epoch}.")
                break

        # SWA finalise
        if swa_active:
            print("Updating SWA BatchNorm stats...")
            update_bn(DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                 shuffle=False, collate_fn=collate_rrc),
                      swa_model, device=self.device)
            swa_m = self.evaluate(dev_dataset, model_override=swa_model)
            if swa_m["macro_f1"] > best_f1:
                best_f1    = swa_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print(f"  SWA improved: macro_f1={best_f1:.4f}")

        total_time = time.time() - t_start
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(MAIN_OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        if best_state:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
            print(f"\nBest model saved to {BEST_MODEL_DIR}/")

        return hist_df, total_time

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()
        swa_ep = None
        if "swa_active" in hist_df.columns:
            rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not rows.empty:
                swa_ep = int(rows.iloc[0])

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(epochs, hist_df["train_loss"], label="Train", marker="o", markersize=3)
        axes[0].plot(epochs, hist_df["val_loss"],   label="Val",   marker="s", markersize=3)
        if swa_ep:
            axes[0].axvline(swa_ep, color="green", linestyle="--", alpha=0.5, label=f"SWA ep{swa_ep}")
        axes[0].set_title("Loss curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["val_macro_f1"], label="Macro-F1", marker="o", markersize=3)
        axes[1].plot(epochs, hist_df["val_rare_f1"],  label="Rare-F1",  marker="^", markersize=3)
        if swa_ep:
            axes[1].axvline(swa_ep, color="green", linestyle="--", alpha=0.5)
        axes[1].set_title("Validation F1"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, "training_curves.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        ax.set_title(f"{split_name.capitalize()} confusion matrix (red = rare)")
        plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_per_class_f1(per_class, split_name, rare_labels):
        f1s    = [per_class[l]["f1"] for l in LABELS]
        colors = ["tomato" if l in rare_labels else "steelblue" for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
        ax.set_title(f"{split_name.capitalize()} per-class F1 (red = rare)")
        ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")


# ═══════════════════════════════════════════════════════════
# PRINT SUMMARY TABLE
# ═══════════════════════════════════════════════════════════
def print_summary(dev_m, test_m, total_time, best_layer_idx):
    keys = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare macro-F1",      "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "="*68)
    print(f"FINAL RESULTS  [layer-selective training, best probe layer={best_layer_idx}]")
    print("="*68)
    print(f"  Total training time : {total_time/60:.2f} min")
    print("-"*68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-"*68)
    for label, key in keys:
        sep = "─"*68 if key == "rare_f1" else ""
        if sep: print(sep)
        print(f"  {label:<28} {dev_m[key]:>12.4f} {test_m[key]:>12.4f}")
    print("="*68)
    print("\n  PER-CLASS F1")
    print("  " + "-"*55)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<20} dev={dv['f1']:.4f}  test={ts['f1']:.4f}  "
              f"prec={ts['precision']:.4f}  rec={ts['recall']:.4f}")
    print("  " + "-"*55)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--phase", choices=["probe", "train", "both"],
                        default="both",
                        help="probe=Phase1 only | train=Phase2 only | both=end-to-end")
    #args = parser.parse_args()
    args, _ = parser.parse_known_args()

    # ── Load data ──────────────────────────────────────────
    print("Loading data...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"Train={len(train_docs)}  Dev={len(dev_docs)}  Test={len(test_docs)}")

    rare_labels, rare_ids, _ = detect_rare_classes(train_docs)
    class_weights             = compute_class_weights(train_docs)

    # ── Phase 1 — probing ─────────────────────────────────
    best_layer_hs_idx = None

    if args.phase in ("probe", "both"):
        best_layer_hs_idx = run_probing_phase(
            train_docs, dev_docs, rare_ids, rare_labels
        )

    if args.phase == "probe":
        print("\nPhase 1 complete. Run with --phase train to proceed.")
        return

    # ── Load probe results if train-only ──────────────────
    if best_layer_hs_idx is None:
        probe_path = os.path.join(PROBE_DIR, "layer_probe_results.json")
        if os.path.exists(probe_path):
            with open(probe_path) as f:
                probe_data = json.load(f)
            best_layer_hs_idx = probe_data["best_layer_hidden_states_idx"]
            print(f"\nLoaded probe results. Best layer hs_idx={best_layer_hs_idx} "
                  f"({probe_data['best_bert_layer_name']})")
        else:
            # Sensible fallback: use layer 8 (index 9 in hidden_states)
            best_layer_hs_idx = 9
            print(f"\nNo probe results found. Using fallback hs_idx={best_layer_hs_idx}")

    # ── Phase 2 — main training ───────────────────────────
    print("\n" + "="*60)
    print("PHASE 2 — LAYER-SELECTIVE MAIN TRAINING")
    print("="*60)

    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    model = InLegalBERT_LayerSelective(
        best_layer_hs_idx = best_layer_hs_idx,
    )

    # Count parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\nParameters — trainable: {trainable:,}  frozen: {frozen:,}\n")

    trainer = Trainer(model, class_weights, rare_ids)

    hist_df, total_time = trainer.train(
        train_dataset, dev_dataset, train_docs, tokenizer
    )
    print("\nTraining complete.")

    # Load best checkpoint
    ckpt = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print("Loaded best checkpoint.")

    # ── Dev evaluation ────────────────────────────────────
    print("\nEvaluating dev set...")
    dev_m = trainer.evaluate(dev_dataset, split_name="dev", measure_time=True)
    with open(os.path.join(MAIN_OUT_DIR, "dev_report.txt"), "w") as f:
        f.write(f"Best probe layer hs_idx={best_layer_hs_idx}\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(dev_m["cls_report"])
    trainer.save_confusion_matrix(dev_m["cm"], "dev", rare_labels)
    trainer.save_per_class_f1(dev_m["per_class_metrics"], "dev", rare_labels)

    # ── Test evaluation ───────────────────────────────────
    print("\nEvaluating test set...")
    test_m = trainer.evaluate(test_dataset, split_name="test", measure_time=True)
    with open(os.path.join(MAIN_OUT_DIR, "test_report.txt"), "w") as f:
        f.write(f"Best probe layer hs_idx={best_layer_hs_idx}\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(test_m["cls_report"])
    trainer.save_confusion_matrix(test_m["cm"], "test", rare_labels)
    trainer.save_per_class_f1(test_m["per_class_metrics"], "test", rare_labels)

    # ── Save predictions & metrics ────────────────────────
    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(MAIN_OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","rare_precision",
        "macro_recall","rare_recall","accuracy",
    ]
    summary = {
        "probing": {
            "best_layer_hs_idx": best_layer_hs_idx,
            "best_bert_layer":   f"layer_{best_layer_hs_idx - 1}",
            "probe_boosted_bert_layers": model.probe_boosted,
        },
        "training": {
            "trainable_params": trainable,
            "frozen_params":    frozen,
            "total_time_min":   total_time / 60,
        },
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "rare_classes": rare_labels,
    }
    with open(os.path.join(MAIN_OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_summary(dev_m, test_m, total_time, best_layer_hs_idx)

    print(f"\nOutputs saved to:")
    print(f"  {PROBE_DIR}/  (probe charts + JSON)")
    print(f"  {MAIN_OUT_DIR}/  (training curves, reports, model)")


if __name__ == "__main__":
    main()

Loading data...
Train=245  Dev=30  Test=50

Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', 'STA', 'PRE_RELIED', 'PRE_NOT_RELIED', 'RATIO', 'RPC', 'NONE']


PHASE 1 — BERT LAYER PROBING
Probing 8 epochs per layer, metric = rare-class macro-F1



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Probing layer  0 ...
           layer  0 | rare_f1=0.3114 | macro_f1=0.3458
  Probing layer  1 ...
           layer  1 | rare_f1=0.3106 | macro_f1=0.3444
  Probing layer  2 ...
           layer  2 | rare_f1=0.3205 | macro_f1=0.3607
  Probing layer  3 ...
           layer  3 | rare_f1=0.3263 | macro_f1=0.3616
  Probing layer  4 ...
           layer  4 | rare_f1=0.3382 | macro_f1=0.3693
  Probing layer  5 ...
           layer  5 | rare_f1=0.3542 | macro_f1=0.3949
  Probing layer  6 ...
           layer  6 | rare_f1=0.3659 | macro_f1=0.4096
  Probing layer  7 ...
           layer  7 | rare_f1=0.3627 | macro_f1=0.4055
  Probing layer  8 ...
           layer  8 | rare_f1=0.3643 | macro_f1=0.4041
  Probing layer  9 ...
           layer  9 | rare_f1=0.3729 | macro_f1=0.4131
  Probing layer 10 ...
           layer 10 | rare_f1=0.3674 | macro_f1=0.4071
  Probing layer 11 ...
           layer 11 | rare_f1=0.3671 | macro_f1=0.4093
  Probing layer 12 ...
           layer 12 | rare_f1=0.3602 | ma

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Layer-selective freeze (probe best = layer 8):
  Frozen  BERT layers : [0, 1, 2, 3, 4, 5]
  Trainable BERT layers : [6, 7, 8, 9, 10, 11]
  (probe-boosted layers : [7, 8, 9])


Parameters — trainable: 44,409,168  frozen: 66,364,416

Optimizer LR schedule:
  bert_pooler                         lr=2.00e-05
  bert_layer_6                        lr=1.18e-05
  bert_layer_7_PROBE_ADJ              lr=5.00e-05
  bert_layer_8_PROBE_BEST             lr=1.00e-04
  bert_layer_9_PROBE_ADJ              lr=5.00e-05
  bert_layer_10                       lr=1.80e-05
  bert_layer_11                       lr=2.00e-05
  head                                lr=5.00e-04

SWA starts at epoch 51/60
Epoch 001/60 | train=299.0347 | val=203.5946 | macro_f1=0.0391 | rare_f1=0.0000 | acc=0.3412 | ES=0/10
  ✔ New best val_macro_f1=0.0391
Epoch 002/60 | train=224.5289 | val=147.1915 | macro_f1=0.1694 | rare_f1=0.0119 | acc=0.6104 | ES=0/10
  ✔ New best val_macro_f1=0.1694
Epoch 003/60 | train=170.5807 | val=110.2908 

In [3]:
# bert_layer_probe_and_train_v2.py
#
# WHAT THIS DOES
# ══════════════
# Phase 1 — Layer Probing
#   Attach a tiny linear classifier (probe) to the mean-pooled output of
#   every BERT layer (0 through N-1 plus the embedding layer).
#   Train each probe independently for a few epochs with BERT fully frozen.
#   Measure rare-class macro-F1 per layer on the dev set.
#
# Phase 2 — Layer-Selective Main Training  (ANTI-OVERFITTING REVISION)
#   Key fixes vs v1:
#     1.  Stochastic Weight Averaging (SWA) for smoother final weights.
#     2.  Cosine LR schedule (was flat / linear).
#     3.  Focal loss (γ=2) + weighted CE + CRF  —  rare-class focused.
#     4.  Inverse-frequency class weights fed into CE & focal loss.
#     5.  WeightedRandomSampler oversamples documents with rare sentences.
#     6.  Dropout raised to 0.3 (matching v3 baseline).
#     7.  Weight decay 0.05 (matching v3 baseline).
#     8.  Gradient accumulation (2 steps) for stable gradients.
#     9.  Label smoothing 0.05 on aux-CE.
#    10.  Early stopping on val_macro_f1 (patience=10).
#    11.  Layer-selective freeze + per-layer LR driven by probing result.
#    12.  SENT_LSTM_LAYERS=1  (2 was over-parameterised given short seqs).
#    13.  CTX_LSTM_LAYERS=1   (same reason).
#    14.  BASE_FREEZE_BELOW=6 — freeze bottom half of BERT always.
#    15.  ALWAYS_UNFREEZE_TOP_N=2 — top 2 layers always trainable.
#    16.  Focal gamma=2.0 + rare_oversample=3.0  (v2 additions for PRE_NOT_RELIED).
#
# USAGE (Jupyter — argparse fix included)
# ═════════════════════════════════════════
#   Phase 1 only :  set PHASE = "probe"  below, or pass --phase probe
#   Phase 2 only :  set PHASE = "train"  below, or pass --phase train
#   Both phases  :  set PHASE = "both"   below, or pass --phase both
#
#   Outputs:
#     probe_results/layer_probe_results.json
#     probe_results/layer_rare_f1_chart.png
#     rrc_layer_selective_v2_logs/  (history, curves, model, reports)

# ── Jupyter-safe phase override ──────────────────────────────────────────────
# If running inside Jupyter, set this string instead of using --phase flag.
# Ignored when the script is run from the terminal (CLI arg takes priority).
JUPYTER_PHASE_OVERRIDE = "both"   # "probe" | "train" | "both"
# ─────────────────────────────────────────────────────────────────────────────

import os, json, random, argparse, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG — shared
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH     = "dataset/build_train.jsonl"
DEV_PATH       = "dataset/build_dev.jsonl"
TEST_PATH      = "dataset/build_test.jsonl"
PROBE_DIR      = "probe_results"
MAIN_OUT_DIR   = "rrc_layer_selective_v2_logs"
BEST_MODEL_DIR = os.path.join(MAIN_OUT_DIR, "best_model")

for _d in [PROBE_DIR, MAIN_OUT_DIR, BEST_MODEL_DIR]:
    os.makedirs(_d, exist_ok=True)

SEED           = 42
MAX_SEQ_LENGTH = 32
RARE_THRESHOLD = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# ═══════════════════════════════════════════════════════════
# CONFIG — Phase 1 (probing)   ← identical to v1
# ═══════════════════════════════════════════════════════════
PROBE_EPOCHS       = 8
PROBE_LR           = 1e-3
PROBE_BATCH        = 32
PROBE_WEIGHT_DECAY = 1e-4

# ═══════════════════════════════════════════════════════════
# CONFIG — Phase 2 (main training)
# ═══════════════════════════════════════════════════════════
# — from layer-probe script (v1) —
BATCH_DOCS     = 2
NUM_EPOCHS     = 60
BERT_LR        = 2e-5
HEAD_LR        = 5e-4
WEIGHT_DECAY   = 0.05      # strong L2 (v2/v3 value)
GRAD_CLIP      = 1.0
DROPOUT        = 0.3
BERT_LR_DECAY  = 0.9

BEST_LAYER_LR_MULTIPLIER     = 5.0
ADJACENT_LAYER_LR_MULTIPLIER = 2.5
ALWAYS_UNFREEZE_TOP_N        = 2
BASE_FREEZE_BELOW            = 6

# — from anti-overfitting v2 —
SENT_LSTM_HIDDEN = 128     # output 256 (bidirectional)
SENT_LSTM_LAYERS = 1       # was 2 → reduced to 1 (short seqs, less overfit)
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64      # output 128
CTX_LSTM_LAYERS  = 1       # was 2 → reduced to 1

AUX_CE_WEIGHT   = 0.5      # weight on aux CE loss alongside CRF
FOCAL_GAMMA     = 2.0      # focal loss γ (punishes easy examples less)
LABEL_SMOOTHING = 0.05
RARE_OVERSAMPLE = 3.0      # oversample weight for docs with rare labels

SWA_START_FRAC  = 0.75     # start SWA at 75% of training (earlier than v1's 85%)
SWA_LR          = 5e-5
ES_PATIENCE     = 12       # slightly more patience than v1
ES_MIN_DELTA    = 1e-4
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_RATIO    = 0.05

# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# DATA UTILITIES
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                out.append(json.loads(s))
    return out


def extract_docs(raw, max_sents=256):
    all_docs = []
    for doc in raw:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    v = item.get("value", {})
                    t = v.get("text", "").strip()
                    l = v.get("labels", ["NONE"])[0]
                    if t:
                        sents.append(t)
                        labs.append(label2id.get(l, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    freqs   = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    print(f"\nLabel frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        flag = " ← RARE" if freqs[lbl] <= threshold else ""
        print(f"  {lbl:<20} {freqs[lbl]*100:5.2f}%  ({counts.get(label2id[lbl],0):5d}){flag}")

    rare_labels = [id2label[i] for i in range(NUM_LABELS) if freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\nRare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, freqs


def compute_class_weights(docs, device=DEVICE):
    """Inverse-frequency weights — strongly upweights rare classes."""
    all_ids = [lid for _, labs in docs for lid in labs]
    counts  = Counter(all_ids)
    total   = len(all_ids)
    weights = [1.0 / (counts.get(i, 0) / total + 1e-6) for i in range(NUM_LABELS)]
    w = torch.tensor(weights, dtype=torch.float32, device=device)
    return w / w.mean()   # normalise: mean weight = 1


# ═══════════════════════════════════════════════════════════
# FLAT SENTENCE DATASET  (probing)
# ═══════════════════════════════════════════════════════════
class FlatSentenceDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.items     = [(s, l) for sents, labs in docs for s, l in zip(sents, labs)]
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        sent, label = self.items[idx]
        enc = self.tokenizer(
            sent, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])).squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }


# ═══════════════════════════════════════════════════════════
# DOCUMENT DATASET  (main training)
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs      = docs
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L, B  = batch[0]["input_ids"].shape[1], len(batch)
    ids   = torch.zeros(B, T_max, L, dtype=torch.long)
    mask  = torch.zeros(B, T_max, L, dtype=torch.long)
    types = torch.zeros(B, T_max, L, dtype=torch.long)
    labs  = torch.full((B, T_max), -100, dtype=torch.long)
    lens  = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        ids[i, :t]   = b["input_ids"]
        mask[i, :t]  = b["attention_mask"]
        types[i, :t] = b["token_type_ids"]
        labs[i, :t]  = b["labels"]
        lens[i]      = t
    return ids, mask, types, labs, lens


def make_oversampled_loader(dataset, docs, rare_ids, batch_size):
    """WeightedRandomSampler: oversample documents containing rare sentences."""
    doc_weights = [
        RARE_OVERSAMPLE if any(l in rare_ids for l in labs) else 1.0
        for _, labs in docs
    ]
    sampler = WeightedRandomSampler(doc_weights, len(doc_weights), replacement=True)
    return DataLoader(dataset, batch_size=batch_size,
                      sampler=sampler, collate_fn=collate_rrc)


# ═══════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if not valid.any():
            return logits.sum() * 0.0
        lv, tv = logits[valid], targets[valid]
        log_p  = F.log_softmax(lv, dim=-1)
        log_pt = log_p.gather(1, tv.unsqueeze(1)).squeeze(1)
        pt     = log_pt.exp()
        focal  = -((1 - pt) ** self.gamma) * log_pt
        if self.weight is not None:
            focal = focal * self.weight[tv]
        return focal.mean()


# ═══════════════════════════════════════════════════════════
# PHASE 1 — LAYER PROBE
# ═══════════════════════════════════════════════════════════
class LayerProbe(nn.Module):
    def __init__(self, hidden_size, num_labels):
        super().__init__()
        self.fc = nn.Linear(hidden_size, num_labels)

    def forward(self, x):
        return self.fc(x)


def extract_layer_representations(bert_model, tokenizer, docs, layer_idx,
                                   batch_size=PROBE_BATCH, device=DEVICE):
    bert_model.eval()
    dataset = FlatSentenceDataset(docs, tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embs, all_labs = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            types = batch["token_type_ids"].to(device)
            labs  = batch["label"]
            out   = bert_model(input_ids=ids, attention_mask=mask,
                               token_type_ids=types, output_hidden_states=True)
            hidden = out.hidden_states[layer_idx]
            mask_f = mask.unsqueeze(-1).float()
            emb    = (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)
            all_embs.append(emb.cpu())
            all_labs.append(labs)
    return torch.cat(all_embs), torch.cat(all_labs)


def probe_one_layer(bert_model, tokenizer, train_docs, dev_docs,
                    layer_idx, rare_ids, class_weights, device=DEVICE):
    print(f"  Probing layer {layer_idx:2d} ...")
    train_embs, train_labs = extract_layer_representations(
        bert_model, tokenizer, train_docs, layer_idx, device=device)
    dev_embs, dev_labs = extract_layer_representations(
        bert_model, tokenizer, dev_docs, layer_idx, device=device)

    probe = LayerProbe(train_embs.shape[-1], NUM_LABELS).to(device)
    opt   = torch.optim.Adam(probe.parameters(), lr=PROBE_LR,
                             weight_decay=PROBE_WEIGHT_DECAY)
    ce    = nn.CrossEntropyLoss(weight=class_weights.to(device))
    dl    = DataLoader(torch.utils.data.TensorDataset(train_embs, train_labs),
                       batch_size=256, shuffle=True)
    best_rare_f1 = 0.0

    for _ in range(PROBE_EPOCHS):
        probe.train()
        for eb, lb in dl:
            eb, lb = eb.to(device), lb.to(device)
            loss = ce(probe(eb), lb)
            opt.zero_grad(); loss.backward(); opt.step()
        probe.eval()
        with torch.no_grad():
            preds = probe(dev_embs.to(device)).argmax(-1).cpu().tolist()
        trues = dev_labs.tolist()
        present_rare = [r for r in rare_ids if r in trues]
        rf1 = f1_score(trues, preds, labels=present_rare,
                       average="macro", zero_division=0) if present_rare else 0.0
        if rf1 > best_rare_f1:
            best_rare_f1 = rf1

    macro_f1 = f1_score(trues, preds, average="macro", zero_division=0)
    print(f"    layer {layer_idx:2d} | rare_f1={best_rare_f1:.4f} | macro_f1={macro_f1:.4f}")
    return {"layer_idx": layer_idx, "rare_f1": best_rare_f1, "macro_f1": macro_f1}


def run_probing_phase(train_docs, dev_docs, rare_ids, rare_labels):
    print("\n" + "="*60)
    print("PHASE 1 — BERT LAYER PROBING")
    print("="*60)
    tokenizer  = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    bert_model = AutoModel.from_pretrained(INLEGALBERT_MODEL_NAME,
                                           output_hidden_states=True).to(DEVICE)
    for p in bert_model.parameters():
        p.requires_grad = False

    class_weights = compute_class_weights(train_docs)
    n_layers      = bert_model.config.num_hidden_layers
    all_results   = []

    for idx in range(0, n_layers + 1):
        r = probe_one_layer(bert_model, tokenizer, train_docs, dev_docs,
                            idx, rare_ids, class_weights)
        r["bert_layer_name"] = "embedding" if idx == 0 else f"layer_{idx-1}"
        all_results.append(r)

    best = sorted(all_results, key=lambda x: x["rare_f1"], reverse=True)[0]
    print(f"\n{'='*60}")
    print(f"Best layer  hs_idx={best['layer_idx']}  ({best['bert_layer_name']})"
          f"  rare_f1={best['rare_f1']:.4f}  macro_f1={best['macro_f1']:.4f}")
    print(f"{'='*60}\n")

    out = {
        "best_layer_hidden_states_idx": best["layer_idx"],
        "best_bert_layer_name":         best["bert_layer_name"],
        "best_rare_f1":                 best["rare_f1"],
        "all_results":                  all_results,
        "rare_labels":                  rare_labels,
        "timestamp":                    datetime.utcnow().isoformat(),
    }
    path = os.path.join(PROBE_DIR, "layer_probe_results.json")
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved probe results → {path}")
    _plot_probe_results(all_results, rare_labels)
    return best["layer_idx"]


def _plot_probe_results(results, rare_labels):
    names    = [r["bert_layer_name"] for r in results]
    rare_f1s = [r["rare_f1"]  for r in results]
    mac_f1s  = [r["macro_f1"] for r in results]
    best_idx = int(np.argmax(rare_f1s))
    x        = np.arange(len(names))
    colors   = ["#EF9F27" if i == best_idx else "#85B7EB" for i in range(len(names))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x, rare_f1s, color=colors, edgecolor="white")
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    axes[0].set_title("Rare-class macro-F1 per BERT layer\n(orange = best)")
    axes[0].set_ylabel("Rare-class macro-F1")
    axes[0].axhline(max(rare_f1s), color="darkorange", linestyle="--", alpha=0.5)
    axes[0].grid(True, alpha=0.3, axis="y")

    axes[1].bar(x, mac_f1s, color="#B5D4F4", edgecolor="white")
    axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    axes[1].set_title("Overall macro-F1 per BERT layer")
    axes[1].set_ylabel("Macro-F1")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.suptitle(f"Layer Probing — InLegalBERT | Rare: {rare_labels}", fontsize=9)
    plt.tight_layout()
    p = os.path.join(PROBE_DIR, "layer_rare_f1_chart.png")
    plt.savefig(p, dpi=150); plt.close()
    print(f"Saved probe chart → {p}")


# ═══════════════════════════════════════════════════════════
# PHASE 2 — ARCHITECTURE
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query      = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        w = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            w = w.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        w       = self.attn_drop(F.softmax(w, dim=-1))
        context = torch.matmul(w, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


class InLegalBERT_v2(nn.Module):
    """
    InLegalBERT + Sent-BiLSTM + MHA-pool + Ctx-BiLSTM + CRF
    with layer-selective freeze driven by probing result.

    Anti-overfitting vs v1:
      • LSTM layers reduced to 1 (short sentences, smaller capacity)
      • Dropout = 0.3  (same as v3 baseline)
      • Weight decay = 0.05 via optimiser
      • SWA + cosine LR (in Trainer)
      • Focal loss + inverse-freq class weights (in Trainer)
      • Label smoothing 0.05 on aux-CE
    """

    def __init__(self, best_layer_hs_idx: int,
                 bert_model_name=INLEGALBERT_MODEL_NAME):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(DROPOUT)

        self.best_hs_idx    = best_layer_hs_idx
        self.best_bert_lidx = max(0, best_layer_hs_idx - 1)
        self._apply_layer_selective_freeze()

        sent_out = SENT_LSTM_HIDDEN * 2   # 256
        ctx_out  = CTX_LSTM_HIDDEN  * 2   # 128

        self.sent_bilstm = nn.LSTM(self.bert_dim, SENT_LSTM_HIDDEN,
                                   num_layers=SENT_LSTM_LAYERS, bidirectional=True,
                                   batch_first=True)
        self.mha_pooling     = MultiHeadAttentionPooling(sent_out, MHA_HEADS, MHA_DROPOUT)
        self.sent_layer_norm = nn.LayerNorm(sent_out)

        self.ctx_bilstm = nn.LSTM(sent_out, CTX_LSTM_HIDDEN,
                                  num_layers=CTX_LSTM_LAYERS, bidirectional=True,
                                  batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_out, ctx_out // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_out // 2, NUM_LABELS),
        )
        self.crf = CRF(num_tags=NUM_LABELS, batch_first=True)

    # ── freeze strategy ───────────────────────────────────
    def _apply_layer_selective_freeze(self):
        n      = len(self.bert.encoder.layer)
        best   = self.best_bert_lidx
        top2   = set(range(n - ALWAYS_UNFREEZE_TOP_N, n))
        probe3 = {max(0, best-1), best, min(n-1, best+1)}
        train_set = top2 | probe3 | set(range(BASE_FREEZE_BELOW, n))

        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        trained, frozen = [], []
        for i, layer in enumerate(self.bert.encoder.layer):
            req = i in train_set
            for p in layer.parameters():
                p.requires_grad = req
            (trained if req else frozen).append(i)

        print(f"\nLayer-selective freeze (probe best = encoder layer {best}):")
        print(f"  Frozen     : {frozen}")
        print(f"  Trainable  : {trained}")
        print(f"  Probe-boost: {sorted(probe3)}\n")
        self.trainable_layer_ids = trained
        self.probe_boosted       = sorted(probe3)

    # ── optimizer with per-layer LR ───────────────────────
    def build_optimizer(self):
        n    = len(self.bert.encoder.layer)
        best = self.best_bert_lidx
        adj  = {max(0, best-1), min(n-1, best+1)} - {best}
        groups = []

        # Pooler
        groups.append({"params": list(self.bert.pooler.parameters()),
                        "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
                        "name": "bert_pooler"})
        # Encoder layers
        for i in self.trainable_layer_ids:
            params = [p for p in self.bert.encoder.layer[i].parameters()
                      if p.requires_grad]
            if not params:
                continue
            if i == best:
                lr, tag = BERT_LR * BEST_LAYER_LR_MULTIPLIER, f"layer_{i}_BEST"
            elif i in adj:
                lr, tag = BERT_LR * ADJACENT_LAYER_LR_MULTIPLIER, f"layer_{i}_ADJ"
            else:
                lr, tag = BERT_LR * (BERT_LR_DECAY ** ((n-1) - i)), f"layer_{i}"
            groups.append({"params": params, "lr": lr,
                            "weight_decay": WEIGHT_DECAY, "name": tag})
        # Head
        head_params = (
            list(self.sent_bilstm.parameters())
            + list(self.mha_pooling.parameters())
            + list(self.sent_layer_norm.parameters())
            + list(self.ctx_bilstm.parameters())
            + list(self.classifier.parameters())
            + list(self.crf.parameters())
        )
        groups.append({"params": head_params, "lr": HEAD_LR,
                        "weight_decay": WEIGHT_DECAY, "name": "head"})

        print("Optimizer LR schedule:")
        for g in groups:
            print(f"  {g['name']:<30} lr={g['lr']:.2e}")
        return torch.optim.AdamW(groups)

    # ── sentence encoder ──────────────────────────────────
    def encode_sentences(self, input_ids, attention_mask, token_type_ids):
        B, T, L = input_ids.shape
        N = B * T
        fids, fmask, ftypes = (input_ids.view(N, L),
                                attention_mask.view(N, L),
                                token_type_ids.view(N, L))
        valid = fmask.sum(-1) > 0
        embs  = fids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(input_ids=fids[valid], attention_mask=fmask[valid],
                            token_type_ids=ftypes[valid])
            embs[valid] = out.last_hidden_state.to(embs.dtype)
        embs = self.dropout(embs)
        lo, _ = self.sent_bilstm(embs)
        lo    = self.dropout(lo)
        pm    = (fmask == 0).clone()
        pm[~valid] = False
        sv = self.mha_pooling(lo, key_padding_mask=pm)
        sv = self.sent_layer_norm(sv)
        sv = sv * valid.unsqueeze(-1).to(sv.dtype)
        return sv.view(B, T, -1)

    # ── forward ───────────────────────────────────────────
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None,
                class_weights=None, focal_loss_fn=None):
        sv = self.dropout(
            self.encode_sentences(input_ids, attention_mask, token_type_ids))

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sv, lengths.cpu(), batch_first=True, enforce_sorted=False)
            po, _ = self.ctx_bilstm(packed)
            ctx, _ = nn.utils.rnn.pad_packed_sequence(po, batch_first=True)
        else:
            ctx, _ = self.ctx_bilstm(sv)

        ctx       = self.dropout(ctx)
        emissions = self.classifier(ctx)
        emissions = torch.nan_to_num(emissions, nan=0., posinf=1e4, neginf=-1e4)

        B, T, _ = emissions.shape
        if lengths is not None:
            crf_mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                crf_mask[i, :l] = True
        elif labels is not None:
            crf_mask = (labels != -100)
        else:
            crf_mask = torch.ones(B, T, dtype=torch.bool, device=emissions.device)

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=crf_mask, reduction="mean")
            fe = emissions.reshape(B*T, -1)
            fl = labels.reshape(B*T)
            ce_fn = nn.CrossEntropyLoss(weight=class_weights,
                                        label_smoothing=LABEL_SMOOTHING,
                                        ignore_index=-100)
            ce_loss    = ce_fn(fe, fl)
            focal_loss = focal_loss_fn(fe, fl) if focal_loss_fn else 0.0
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss + 0.3 * focal_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=crf_mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    wp          = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec   = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec   = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    wr          = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc         = accuracy_score(all_trues, all_preds)

    pc_f1  = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                      average=None, zero_division=0)
    pc_pr  = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                             average=None, zero_division=0)
    pc_re  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                          average=None, zero_division=0)
    per_class = {id2label[i]: {"f1": float(pc_f1[i]),
                                "precision": float(pc_pr[i]),
                                "recall": float(pc_re[i])}
                 for i in range(NUM_LABELS)}

    present_rare = [r for r in rare_ids if r in all_trues]
    rare_f1  = f1_score(all_trues, all_preds, labels=present_rare,
                        average="macro", zero_division=0) if present_rare else 0.0
    rare_pr  = precision_score(all_trues, all_preds, labels=present_rare,
                               average="macro", zero_division=0) if present_rare else 0.0
    rare_rec = recall_score(all_trues, all_preds, labels=present_rare,
                            average="macro", zero_division=0) if present_rare else 0.0

    str_t = [id2label[x] for x in all_trues]
    str_p = [id2label[x] for x in all_preds]
    cls_rep = classification_report(str_t, str_p, labels=LABELS, digits=4, zero_division=0)
    cm      = confusion_matrix(str_t, str_p, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": wp,
        "macro_recall": macro_rec, "micro_recall": micro_rec, "weighted_recall": wr,
        "rare_f1": rare_f1, "rare_precision": rare_pr, "rare_recall": rare_rec,
        "per_class_metrics": per_class,
        "accuracy": acc, "cls_report": cls_rep, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience  = patience
        self.min_delta = min_delta
        self.best      = -1.0
        self.counter   = 0
        self.stop      = False

    def step(self, score):
        if score > self.best + self.min_delta:
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PHASE 2 — TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, class_weights, rare_ids, device=DEVICE):
        self.model         = model.to(device)
        self.device        = device
        self.class_weights = class_weights.to(device)
        self.focal_fn      = FocalLoss(gamma=FOCAL_GAMMA,
                                       weight=class_weights.to(device),
                                       ignore_index=-100)
        self.rare_ids = rare_ids

    # ── validation loss ───────────────────────────────────
    def _val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, mask, types, labs, lens in loader:
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                labs, lens = labs.to(self.device), lens.to(self.device)
                loss, _ = self.model(ids, mask, types, labels=labs, lengths=lens,
                                     class_weights=self.class_weights,
                                     focal_loss_fn=self.focal_fn)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    # ── evaluation ────────────────────────────────────────
    def evaluate(self, dataset, split_name="dev",
                 measure_time=False, model_override=None):
        m = model_override or self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_time else None
        with torch.no_grad():
            for ids, mask, types, labs, lens in loader:
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                lens = lens.to(self.device)
                decoded, _ = m(ids, mask, types, labels=None, lengths=lens)
                for i, seq in enumerate(decoded):
                    tl = int(lens[i].item())
                    all_preds.extend(seq)
                    all_trues.extend(labs[i, :tl].tolist())
        if measure_time:
            el = time.time() - t0
            print(f"  Inference ({split_name}): {el:.2f}s | "
                  f"{len(all_trues)/el:.0f} sent/s")
        return compute_all_metrics(all_trues, all_preds, self.rare_ids)

    # ── training loop ─────────────────────────────────────
    def train(self, train_dataset, dev_dataset, train_docs, tokenizer):
        loader      = make_oversampled_loader(train_dataset, train_docs,
                                              self.rare_ids, BATCH_DOCS)
        optimizer   = self.model.build_optimizer()
        total_steps = len(loader) * NUM_EPOCHS // GRADIENT_ACCUMULATION_STEPS
        warmup_steps= int(WARMUP_RATIO * total_steps)
        scheduler   = get_cosine_schedule_with_warmup(optimizer, warmup_steps,
                                                       total_steps)

        swa_model  = AveragedModel(self.model)
        swa_start  = max(1, int(NUM_EPOCHS * SWA_START_FRAC))
        swa_sched  = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
        swa_active = False

        es         = EarlyStopping()
        best_f1    = -1.0
        best_state = None
        history    = []
        t_start    = time.time()

        print(f"\nSWA starts at epoch {swa_start}/{NUM_EPOCHS}")
        print(f"Early stopping patience = {ES_PATIENCE}\n")

        for epoch in range(1, NUM_EPOCHS + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            optimizer.zero_grad()

            for step, (ids, mask, types, labs, lens) in enumerate(loader):
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                labs, lens = labs.to(self.device), lens.to(self.device)
                loss, _ = self.model(ids, mask, types, labels=labs, lengths=lens,
                                     class_weights=self.class_weights,
                                     focal_loss_fn=self.focal_fn)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()
                run_loss += loss.item(); n_steps += 1

            # flush leftover gradient
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active: scheduler.step()
                optimizer.zero_grad()

            # SWA
            if epoch >= swa_start:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_sched.step()

            avg_loss    = run_loss / max(1, n_steps)
            val_loss    = self._val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset)

            tag = " [SWA]" if swa_active else ""
            print(
                f"Ep {epoch:03d}/{NUM_EPOCHS} | "
                f"train={avg_loss:.4f} | val={val_loss:.4f} | "
                f"macro_f1={val_metrics['macro_f1']:.4f} | "
                f"rare_f1={val_metrics['rare_f1']:.4f} | "
                f"acc={val_metrics['accuracy']:.4f} | "
                f"ES={es.counter}/{es.patience}{tag}"
            )
            history.append({
                "epoch": epoch, "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1":  val_metrics["macro_f1"],
                "val_rare_f1":   val_metrics["rare_f1"],
                "val_accuracy":  val_metrics["accuracy"],
                "swa_active":    swa_active,
                "gap_macro_f1":  val_metrics["macro_f1"] - (avg_loss * 0),  # placeholder
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if es.step(val_metrics["macro_f1"]):
                print(f"\nEarly stopping at epoch {epoch}.")
                break

        # SWA finalise
        if swa_active:
            print("Updating SWA BatchNorm stats...")
            update_bn(
                DataLoader(train_dataset, batch_size=BATCH_DOCS,
                           shuffle=False, collate_fn=collate_rrc),
                swa_model, device=self.device,
            )
            swa_m = self.evaluate(dev_dataset, model_override=swa_model)
            if swa_m["macro_f1"] > best_f1:
                best_f1    = swa_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print(f"  SWA improved → macro_f1={best_f1:.4f}")

        total_time = time.time() - t_start
        hist_df    = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(MAIN_OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        if best_state:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
            print(f"\nBest model saved → {BEST_MODEL_DIR}/")

        return hist_df, total_time

    # ── plots ─────────────────────────────────────────────
    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()
        swa_ep = None
        if "swa_active" in hist_df.columns:
            rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not rows.empty:
                swa_ep = int(rows.iloc[0])

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Loss
        axes[0].plot(epochs, hist_df["train_loss"], label="Train", marker="o", ms=3)
        axes[0].plot(epochs, hist_df["val_loss"],   label="Val",   marker="s", ms=3)
        if swa_ep:
            axes[0].axvline(swa_ep, color="green", linestyle="--", alpha=0.5,
                            label=f"SWA ep{swa_ep}")
        axes[0].set_title("Loss curves — ideally should converge together\n"
                          "(large gap = overfitting)")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

        # F1
        axes[1].plot(epochs, hist_df["val_macro_f1"], label="Macro-F1", marker="o", ms=3)
        axes[1].plot(epochs, hist_df["val_rare_f1"],  label="Rare-F1",  marker="^", ms=3)
        if swa_ep:
            axes[1].axvline(swa_ep, color="green", linestyle="--", alpha=0.5)
        axes[1].set_title("Validation F1")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")

        # Train/Val loss gap  — overfitting diagnostic
        gap = (hist_df["val_loss"] - hist_df["train_loss"]).clip(lower=0)
        axes[2].fill_between(epochs, gap, alpha=0.4, color="tomato", label="Val−Train gap")
        axes[2].plot(epochs, gap, color="tomato", lw=1.5)
        if swa_ep:
            axes[2].axvline(swa_ep, color="green", linestyle="--", alpha=0.5,
                            label=f"SWA ep{swa_ep}")
        axes[2].set_title("Overfitting gap (val_loss − train_loss)\n"
                          "Should stay close to 0")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss gap")

        plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, "training_curves.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        ax.set_title(f"{split_name.capitalize()} confusion matrix (red = rare)")
        plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

    @staticmethod
    def save_per_class_f1(per_class, split_name, rare_labels):
        f1s    = [per_class[l]["f1"] for l in LABELS]
        colors = ["tomato" if l in rare_labels else "steelblue" for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
        ax.set_title(f"{split_name.capitalize()} per-class F1 (red = rare)")
        ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
        p = os.path.join(MAIN_OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")


# ═══════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═══════════════════════════════════════════════════════════
def print_summary(dev_m, test_m, total_time, best_layer_idx):
    keys = [
        ("Accuracy",        "accuracy"),
        ("Macro-F1",        "macro_f1"),
        ("Micro-F1",        "micro_f1"),
        ("Weighted-F1",     "weighted_f1"),
        ("Rare macro-F1",   "rare_f1"),
        ("Macro-Precision", "macro_precision"),
        ("Rare-Precision",  "rare_precision"),
        ("Macro-Recall",    "macro_recall"),
        ("Rare-Recall",     "rare_recall"),
    ]
    print("\n" + "="*68)
    print(f"FINAL RESULTS  [v2 anti-overfitting | probe layer={best_layer_idx}]")
    print("="*68)
    print(f"  Total training time : {total_time/60:.2f} min")

    # Overfitting check
    dev_mf1  = dev_m["macro_f1"]
    test_mf1 = test_m["macro_f1"]
    gap      = abs(test_mf1 - dev_mf1)
    status   = "OK (gap ≤ 5%)" if gap <= 0.05 else ("WARN gap 5-10%" if gap <= 0.10
                                                       else "CHECK gap > 10%")
    print(f"  Dev/Test gap        : {gap:.4f}  [{status}]")
    print("-"*68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-"*68)
    for label, key in keys:
        if key == "rare_f1":
            print("─"*68)
        print(f"  {label:<28} {dev_m[key]:>12.4f} {test_m[key]:>12.4f}")
    print("="*68)
    print("\n  PER-CLASS F1")
    print("  " + "-"*55)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]
        ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<20} dev={dv['f1']:.4f}  test={ts['f1']:.4f}  "
              f"prec={ts['precision']:.4f}  rec={ts['recall']:.4f}")
    print("  " + "-"*55)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    # ── Jupyter-safe argument parsing ────────────────────
    parser = argparse.ArgumentParser()
    parser.add_argument("--phase", choices=["probe", "train", "both"],
                        default=None,
                        help="probe | train | both  (default: use JUPYTER_PHASE_OVERRIDE)")
    args, _ = parser.parse_known_args()   # ignore Jupyter kernel args

    # If no CLI phase provided, fall back to the constant at the top of the file
    phase = args.phase if args.phase is not None else JUPYTER_PHASE_OVERRIDE

    print(f"Running phase: {phase}")
    print(f"Device: {DEVICE}")
    print("="*60)
    print("Key anti-overfitting settings:")
    print(f"  Dropout              : {DROPOUT}")
    print(f"  Weight decay         : {WEIGHT_DECAY}")
    print(f"  LSTM layers (sent)   : {SENT_LSTM_LAYERS}  (was 2)")
    print(f"  LSTM layers (ctx)    : {CTX_LSTM_LAYERS}   (was 2)")
    print(f"  SWA start            : {SWA_START_FRAC*100:.0f}% of epochs")
    print(f"  Early stopping       : patience={ES_PATIENCE}")
    print(f"  Focal loss γ         : {FOCAL_GAMMA}")
    print(f"  Rare oversample      : {RARE_OVERSAMPLE}×")
    print(f"  Label smoothing      : {LABEL_SMOOTHING}")
    print(f"  Base freeze below    : layer {BASE_FREEZE_BELOW}")
    print("="*60)

    # ── Load data ─────────────────────────────────────────
    print("\nLoading data...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"Train={len(train_docs)}  Dev={len(dev_docs)}  Test={len(test_docs)}")

    rare_labels, rare_ids, _ = detect_rare_classes(train_docs)
    class_weights             = compute_class_weights(train_docs)

    # ── Phase 1 ───────────────────────────────────────────
    best_layer_hs_idx = None

    if phase in ("probe", "both"):
        best_layer_hs_idx = run_probing_phase(
            train_docs, dev_docs, rare_ids, rare_labels)

    if phase == "probe":
        print("\nPhase 1 complete.  Run with phase='train' to proceed.")
        return

    # ── Load probe results (train-only mode) ──────────────
    if best_layer_hs_idx is None:
        probe_path = os.path.join(PROBE_DIR, "layer_probe_results.json")
        if os.path.exists(probe_path):
            with open(probe_path) as f:
                pd_data = json.load(f)
            best_layer_hs_idx = pd_data["best_layer_hidden_states_idx"]
            print(f"\nLoaded probe results: hs_idx={best_layer_hs_idx} "
                  f"({pd_data['best_bert_layer_name']})")
        else:
            best_layer_hs_idx = 9   # sensible fallback (layer_8)
            print(f"\nNo probe results found. Using fallback hs_idx={best_layer_hs_idx}")

    # ── Phase 2 ───────────────────────────────────────────
    print("\n" + "="*60)
    print("PHASE 2 — LAYER-SELECTIVE MAIN TRAINING  (v2)")
    print("="*60)

    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    model = InLegalBERT_v2(best_layer_hs_idx=best_layer_hs_idx)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"Parameters — trainable: {trainable:,}  frozen: {frozen:,}\n")

    trainer = Trainer(model, class_weights, rare_ids)
    hist_df, total_time = trainer.train(
        train_dataset, dev_dataset, train_docs, tokenizer)
    print("\nTraining complete.")

    # Load best checkpoint
    ckpt = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print("Loaded best checkpoint.")

    # ── Dev eval ──────────────────────────────────────────
    print("\nEvaluating dev set...")
    dev_m = trainer.evaluate(dev_dataset, split_name="dev", measure_time=True)
    with open(os.path.join(MAIN_OUT_DIR, "dev_report.txt"), "w") as f:
        f.write(f"v2 | probe hs_idx={best_layer_hs_idx} | "
                f"rare={rare_labels}\n\n{dev_m['cls_report']}")
    trainer.save_confusion_matrix(dev_m["cm"], "dev", rare_labels)
    trainer.save_per_class_f1(dev_m["per_class_metrics"], "dev", rare_labels)

    # ── Test eval ─────────────────────────────────────────
    print("\nEvaluating test set...")
    test_m = trainer.evaluate(test_dataset, split_name="test", measure_time=True)
    with open(os.path.join(MAIN_OUT_DIR, "test_report.txt"), "w") as f:
        f.write(f"v2 | probe hs_idx={best_layer_hs_idx} | "
                f"rare={rare_labels}\n\n{test_m['cls_report']}")
    trainer.save_confusion_matrix(test_m["cm"], "test", rare_labels)
    trainer.save_per_class_f1(test_m["per_class_metrics"], "test", rare_labels)

    # ── Save predictions & summary ────────────────────────
    pd.DataFrame({
        "true": [id2label[x] for x in test_m["all_trues"]],
        "pred": [id2label[x] for x in test_m["all_preds"]],
    }).to_csv(os.path.join(MAIN_OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "rare_precision",
        "macro_recall", "rare_recall", "accuracy",
    ]
    summary = {
        "version": "v2-anti-overfitting",
        "probing": {
            "best_layer_hs_idx":         best_layer_hs_idx,
            "best_bert_layer":           f"layer_{best_layer_hs_idx - 1}",
            "probe_boosted_bert_layers": model.probe_boosted,
        },
        "training": {
            "trainable_params": trainable,
            "frozen_params":    frozen,
            "total_time_min":   total_time / 60,
        },
        "hyperparams": {
            "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY,
            "sent_lstm_layers": SENT_LSTM_LAYERS, "ctx_lstm_layers": CTX_LSTM_LAYERS,
            "focal_gamma": FOCAL_GAMMA, "rare_oversample": RARE_OVERSAMPLE,
            "label_smoothing": LABEL_SMOOTHING, "swa_start_frac": SWA_START_FRAC,
            "es_patience": ES_PATIENCE, "bert_lr": BERT_LR, "head_lr": HEAD_LR,
        },
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "rare_classes": rare_labels,
        "dev_test_macro_f1_gap": abs(dev_m["macro_f1"] - test_m["macro_f1"]),
    }
    with open(os.path.join(MAIN_OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_summary(dev_m, test_m, total_time, best_layer_hs_idx)

    print(f"\nAll outputs saved to:")
    print(f"  {PROBE_DIR}/")
    print(f"  {MAIN_OUT_DIR}/")


if __name__ == "__main__":
    main()

Running phase: both
Device: cuda:0
Key anti-overfitting settings:
  Dropout              : 0.3
  Weight decay         : 0.05
  LSTM layers (sent)   : 1  (was 2)
  LSTM layers (ctx)    : 1   (was 2)
  SWA start            : 75% of epochs
  Early stopping       : patience=12
  Focal loss γ         : 2.0
  Rare oversample      : 3.0×
  Label smoothing      : 0.05
  Base freeze below    : layer 6

Loading data...
Train=245  Dev=30  Test=50

Label frequency analysis (threshold ≤ 5%):
  PREAMBLE             14.50%  ( 4167)
  FAC                  19.99%  ( 5744)
  RLC                   2.62%  (  752) ← RARE
  ISSUE                 1.28%  (  367) ← RARE
  ARG_PETITIONER        4.58%  ( 1315) ← RARE
  ARG_RESPONDENT        2.43%  (  698) ← RARE
  ANALYSIS             36.66%  (10537)
  STA                   1.67%  (  481) ← RARE
  PRE_RELIED            4.97%  ( 1427) ← RARE
  PRE_NOT_RELIED        0.55%  (  158) ← RARE
  RATIO                 2.30%  (  661) ← RARE
  RPC                   3.67%  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Probing layer  0 ...
    layer  0 | rare_f1=0.3114 | macro_f1=0.3458
  Probing layer  1 ...
    layer  1 | rare_f1=0.3106 | macro_f1=0.3444
  Probing layer  2 ...
    layer  2 | rare_f1=0.3205 | macro_f1=0.3607
  Probing layer  3 ...
    layer  3 | rare_f1=0.3263 | macro_f1=0.3616
  Probing layer  4 ...
    layer  4 | rare_f1=0.3382 | macro_f1=0.3693
  Probing layer  5 ...
    layer  5 | rare_f1=0.3542 | macro_f1=0.3949
  Probing layer  6 ...
    layer  6 | rare_f1=0.3659 | macro_f1=0.4096
  Probing layer  7 ...
    layer  7 | rare_f1=0.3627 | macro_f1=0.4055
  Probing layer  8 ...
    layer  8 | rare_f1=0.3643 | macro_f1=0.4041
  Probing layer  9 ...
    layer  9 | rare_f1=0.3729 | macro_f1=0.4131
  Probing layer 10 ...
    layer 10 | rare_f1=0.3674 | macro_f1=0.4071
  Probing layer 11 ...
    layer 11 | rare_f1=0.3671 | macro_f1=0.4093
  Probing layer 12 ...
    layer 12 | rare_f1=0.3602 | macro_f1=0.3948

Best layer  hs_idx=9  (layer_8)  rare_f1=0.3729  macro_f1=0.4131

Saved prob

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Layer-selective freeze (probe best = encoder layer 8):
  Frozen     : [0, 1, 2, 3, 4, 5]
  Trainable  : [6, 7, 8, 9, 10, 11]
  Probe-boost: [7, 8, 9]

Parameters — trainable: 44,409,168  frozen: 66,364,416

Optimizer LR schedule:
  bert_pooler                    lr=2.00e-05
  layer_6                        lr=1.18e-05
  layer_7_ADJ                    lr=5.00e-05
  layer_8_BEST                   lr=1.00e-04
  layer_9_ADJ                    lr=5.00e-05
  layer_10                       lr=1.80e-05
  layer_11                       lr=2.00e-05
  head                           lr=5.00e-04

SWA starts at epoch 45/60
Early stopping patience = 12

Ep 001/60 | train=299.0347 | val=203.5946 | macro_f1=0.0391 | rare_f1=0.0000 | acc=0.3412 | ES=0/12
  ✔ New best val_macro_f1=0.0391
Ep 002/60 | train=224.5289 | val=147.1915 | macro_f1=0.1694 | rare_f1=0.0119 | acc=0.6104 | ES=0/12
  ✔ New best val_macro_f1=0.1694
Ep 003/60 | train=170.5807 | val=110.2908 | macro_f1=0.2440 | rare_f1=0.0839 | acc=0.6